# Prerequisites

These steps must be completed before running this notebook. Follow the instructions below to ensure your environment is properly set up and ready to execute the workflow.


## Step 1: Install Docker (Skip this is you have Docker Running)

Docker is required to run Oracle in a containerized environment. Install Docker for your platform:

**macOS:**
1. Download Docker Desktop from https://www.docker.com/products/docker-desktop/
2. Install the `.dmg` file
3. Open Docker Desktop from Applications
4. Wait for Docker to start (whale icon in menu bar)
5. Verify: Open Terminal and run `docker ps` (should not error)

**Windows:**
1. Download Docker Desktop from https://www.docker.com/products/docker-desktop/
2. Run the installer
3. Follow the setup wizard (may require WSL 2)
4. Launch Docker Desktop from Start menu
5. Wait for Docker to start
6. Verify: Open PowerShell/CMD and run `docker ps` (should not error)

**Linux (Ubuntu/Debian):**
```bash
# Update package index
sudo apt-get update

# Install Docker
sudo apt-get install -y docker.io

# Start Docker service
sudo systemctl start docker
sudo systemctl enable docker

# Add your user to docker group (optional, to run without sudo)
sudo usermod -aG docker $USER
# Log out and back in for group changes to take effect

# Verify installation
docker ps
```

**Verify Docker is working:**
```bash
docker --version
docker ps
```

If both commands work, Docker is installed and running.

---

## Step 2: Get MemoRizz (if not using CLI)

If you prefer not to use the `memorizz` CLI command, you can clone the repository to access the installation script:

```bash
# Clone the repository
git clone https://github.com/RichmondAlake/memorizz.git
cd memorizz

# Make the installation script executable
chmod +x install_oracle.sh
```

**Note:** If you're using `pip install memorizz[oracle]`, you can still use the CLI commands (`memorizz install-oracle` and `memorizz setup-oracle`) without cloning the repo. The CLI will work for pip-installed users.

---

### Summary

Before proceeding, ensure:
- ✅ Docker is installed and running
- ✅ Docker is verified working (`docker ps` succeeds)
- ✅ (Optional) Repository cloned if you want to use scripts directly instead of CLI

Once Docker is ready, you can proceed to install Oracle using either:
- `memorizz install-oracle` (CLI - works for pip-installed users)
- `./install_oracle.sh` (Script - requires cloned repo)

# Setup: Package Installation

Install all required packages for this demo:

- **memorizz** - Core MemoRizz library for building AI agents with persistent memory
- **oracledb** - Oracle database driver for connecting to Oracle Database
- **openai** - OpenAI SDK for LLM and embedding API access
- **requests** - HTTP library for making API calls (used in tool examples)
- **python-dotenv** - Loads environment variables from `.env` files for secure credential management


In [1]:
# Install memorizz and required dependencies
%pip install -qU memorizz

# Install Oracle database driver (required for Oracle provider)
%pip install -qU oracledb

# Install OpenAI SDK (for LLM and embeddings)
%pip install -qU openai

# Install requests (for tool examples like weather API)
%pip install -qU requests

# Install python-dotenv for .env file support (optional but recommended)
%pip install -qU python-dotenv

print("✅ All packages installed successfully!")


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
✅ All packages installed successfully!


# Part 1: Oracle AI Database Installation and Setup

In this step, we will provision and install a local Oracle AI Database instance by pulling and running the official Docker image. 

> This containerized deployment provides an isolated environment with full AI and vector-search > capabilities, acting as the Memory Core that MemoRizz and its agent workloads rely on for > persistent storage, retrieval, and indexing.

**There are three ways to install Oracle with MemoRizz**

1. Via the MemoRizz CLI: ```memorizz install-oracle``` (easiest)
2. Via the installation script: ```./install_oracle.sh``` (requires cloning the MemoRizz repo)

Either way you select, 1 and 2 will need to have Docker installed on your machine.

**After installing Oracle, you can set up the database schema using:**
- ```memorizz setup-oracle``` (CLI)
- Or the Python function ```setup_oracle_user()```
- Or the script ```./setup_oracle.sh``` (requires cloning the MemoRizz repo)

### Option 1: Using the MemoRizz CLI (Recommended for getting started)

#### Installing Oracle

The `! memorizz install-oracle` command:

1. **Installs and starts Oracle AI Database Free** in a Docker container on your local machine.

2. **Checks if Docker is running** — exits with an error if Docker isn't available.

3. **Checks for an existing container** — if `oracle-memorizz` exists and is stopped, it starts it; if it's running, it skips; if missing, it creates a new one.

4. **Pulls the Oracle image** (if not already downloaded) — defaults to the `latest-lite` version (~1.78GB).

5. **Creates a persistent Docker volume** (`oracle-memorizz-data`) so your data survives container restarts.

6. **Waits for the database to be ready** — monitors logs until "DATABASE IS READY TO USE!" appears (typically 2–3 minutes).

7. **Displays connection details** — shows host, port, service name, and credentials, and exports environment variables you can use in your shell.

**Note:** The `!` prefix runs the command in a shell from a Jupyter notebook. 

In a terminal, use `memorizz install-oracle` without the `!`.

In [2]:
! memorizz install-oracle

note: `memorizz install-oracle` is now `memorizz oracle install`
🔍 Checking if Docker is running...
✅ Docker is running

╔═══════════════════════════════════════════════════════════════════════╗
║           Select Oracle Database Docker Image                        ║
╚═══════════════════════════════════════════════════════════════════════╝

  1) Official Oracle 23ai Free - Lite Edition (Recommended)
     Image: container-registry.oracle.com/database/free:latest-lite
     Size: ~1.78GB
     Features: AI Vector Search, Full Oracle 23ai capabilities

  2) Official Oracle 23ai Free - Full Edition
     Image: container-registry.oracle.com/database/free:latest
     Size: ~9.93GB
     Features: AI Vector Search, All Oracle 23ai features + extras

  3) Community gvenzl/oracle-free (Faster startup)
     Image: gvenzl/oracle-free:latest
     Size: ~3GB
     Features: Oracle 23ai Free, Optimized for development
     Note: Community-maintained, faster initialization

Enter your choice (1-3) [1]: 


Once the command above completes, you should see information with connecting to your database procvided, this will show the host, port, service name and credentials

This information will need to go in your local environment as shown below

#### Setting Environment variables

In [1]:
import os

ORACLE_ADMIN_PASSWORD = os.getenv("ORACLE_ADMIN_PASSWORD", "MyPassword123!")
ORACLE_USER = "memorizz_user"
ORACLE_PASSWORD = "SecurePass123!"
ORACLE_DSN = "localhost:1521/FREEPDB1"
OPENAI_EMBEDDING_MODEL = "text-embedding-3-small"
OPENAI_EMBEDDING_DIMENSIONS = "256"

os.environ["ORACLE_ADMIN_PASSWORD"] = ORACLE_ADMIN_PASSWORD
os.environ["ORACLE_USER"] = ORACLE_USER
os.environ["ORACLE_PASSWORD"] = ORACLE_PASSWORD
os.environ["ORACLE_DSN"] = ORACLE_DSN
os.environ["MEMORIZZ_DEFAULT_EMBEDDING_PROVIDER"] = "openai"
os.environ["MEMORIZZ_DEFAULT_EMBEDDING_MODEL"] = OPENAI_EMBEDDING_MODEL
os.environ["MEMORIZZ_DEFAULT_EMBEDDING_DIMENSIONS"] = OPENAI_EMBEDDING_DIMENSIONS


#### Setting up Oracle

The `! memorizz setup-oracle` command:

1. **Sets up the database schema** for MemoRizz in your Oracle database.

2. **Creates the `memorizz_user`** if it doesn't exist, with the password from your environment variables.

3. **Grants required privileges** — CREATE SESSION, CREATE TABLE, CREATE VIEW, CREATE SEQUENCE, CREATE TRIGGER, and AI Vector Search privileges (DBMS_VECTOR, DBMS_VECTOR_CHAIN).

4. **Configures the default tablespace** — sets a tablespace with automatic segment space management (required for VECTOR types).

5. **Creates relational tables** — executes `schema_relational.sql` to create tables (AGENTS, PERSONAS, TOOLBOX, CONVERSATION_MEMORY, etc.) with proper indexes.

6. **Creates JSON Duality Views** — executes `duality_views.sql` to create JSON document interfaces over the relational tables.

7. **Verifies the setup** — checks that tables, views, and vector indexes were created successfully.

8. **Displays a summary** — shows counts of created tables, views, and indexes, plus connection details for your application.

**Note:** This assumes Oracle is already installed and running (via `memorizz install-oracle` or manually). The `!` prefix runs the command in a shell from a Jupyter notebook. In a terminal, use `memorizz setup-oracle` without the `!`.

In [ ]:
! memorizz setup-oracle

### Option 2: Manual Installation (Skip this if you went through Option 1)


#### Installation

Before running install_oracle.sh:
1. Start Docker Desktop (or Docker daemon on Linux)
2. Wait for Docker to be fully started (check system tray/status)
3. Then run: ./install_oracle.sh

> To use the installation script you either have to have cloned the repo or, you can get the script here: https://github.com/RichmondAlake/memorizz/blob/main/install_oracle.sh

Run the following command below in a terminal on your local machine

```bash
# Make script executable (if needed)
chmod +x install_oracle.sh

# Install Oracle Database (includes persistent volume)
./install_oracle.sh

# For Apple Silicon (M1/M2/M3):
export PLATFORM_FLAG="--platform linux/amd64"
./install_oracle.sh
```

![Model Architecture](../images/memorizz_script_output.png)

Running the command above (install_oracle.sh script) does the following

1. Starts Oracle Database 23ai Free in a Docker container for local development. Idempotent: safe to run multiple times.
2. Initialization: Sets container name, volume name, and Oracle image; reads password and platform settings from environment variables.
3. Docker Check: Verifies Docker is running; exits with error if not.
4. Container Check: If container exists and is running, skips; if stopped, starts it; if missing, creates a new one.
5. First Run Setup: Pulls Oracle image, creates persistent volume, creates and starts container with port mapping and data persistence.
6. Wait for Ready: Polls logs every 5 seconds until "DATABASE IS READY TO USE!" appears (typically 2-3 minutes).
7. Display Info: Shows connection details (host, port, credentials) and exports environment variables for use in your shell.

The output of a successful execution of the command above will provide you environment variables that you can plug into the next cell below

In [4]:
ORACLE_ADMIN_PASSWORD="${ORACLE_ADMIN_PASSWORD:-MyPassword123!}"
ORACLE_USER="memorizz_user"
ORACLE_PASSWORD="SecurePass123!"
ORACLE_DSN="localhost:1521/FREEPDB1"
MEMORIZZ_DEFAULT_EMBEDDING_PROVIDER="openai"
MEMORIZZ_DEFAULT_EMBEDDING_MODEL="text-embedding-3-small"
MEMORIZZ_DEFAULT_EMBEDDING_DIMENSIONS="256"


In [5]:
import os

os.environ["ORACLE_ADMIN_PASSWORD"] = os.getenv("ORACLE_ADMIN_PASSWORD", "MyPassword123!")
os.environ["ORACLE_USER"] = "memorizz_user"
os.environ["ORACLE_PASSWORD"] = "SecurePass123!"
os.environ["ORACLE_DSN"] = "localhost:1521/FREEPDB1"
os.environ["MEMORIZZ_DEFAULT_EMBEDDING_PROVIDER"] = "openai"
os.environ["MEMORIZZ_DEFAULT_EMBEDDING_MODEL"] = "text-embedding-3-small"
os.environ["MEMORIZZ_DEFAULT_EMBEDDING_DIMENSIONS"] = "256"


In [2]:
# Database connection details
# Option 1: Use environment variables (recommended)
import os
from pathlib import Path

# Try to load from .env file if available
try:
    from dotenv import load_dotenv
    env_path = Path(__file__).parent.parent.parent / ".env"
    load_dotenv(env_path)
    print("✓ Loaded credentials from .env file")
except ImportError:
    print("ℹ python-dotenv not installed. Install with: pip install python-dotenv")
except Exception:
    pass

# Get credentials from environment variables with defaults
ORACLE_USER = os.getenv("ORACLE_USER", "")
ORACLE_PASSWORD = os.getenv("ORACLE_PASSWORD", "")
ORACLE_DSN = os.getenv("ORACLE_DSN", "")

print(f"Using Oracle connection:")
print(f"  User: {ORACLE_USER}")
print(f"  DSN: {ORACLE_DSN}")

Using Oracle connection:
  User: memorizz_user
  DSN: localhost:1521/FREEPDB1


#### Setup

Ways to set up the Oracle database after installing:

1. **Python module** - `python -m memorizz.cli setup-oracle`
2. **Examples script** - `python examples/setup_oracle_user.py`
3. **Python import** - `from memorizz.memory_provider.oracle.setup import setup_oracle_user` then call `setup_oracle_user()`
4. **Manual SQL** - Create user manually via SQL, then run SQL files manually (`schema_relational.sql` and `duality_views.sql`)


In [3]:
from memorizz.memory_provider.oracle.setup import setup_oracle_user
setup_oracle_user()

Oracle Database Complete Setup for Memorizz

✓ Found schema file: schema_relational.sql

Detecting setup mode...
----------------------------------------------------------------------
  Admin user 'system' cannot create users
  Trying SYS as SYSDBA (has full privileges)...
  ✓ Connected as SYS as SYSDBA (has CREATE USER privilege)
✓ Admin mode detected: Full setup with user creation
  Connected as: sys
  Can create users: Yes

STEP 1: Creating User and Granting Privileges
----------------------------------------------------------------------

Dropping existing memorizz_user user (if exists)...
  Checking for active sessions...
  No active sessions found
  ✓ Dropped existing memorizz_user user

Creating memorizz_user user...
  ✓ User memorizz_user created

Granting basic privileges (least-privilege)...
  ✓ CREATE SESSION (required for database connections)
  ✓ CREATE TABLE (required for memory storage tables and indexes)
  ✓ CREATE VIEW (required for Memorizz views)
  ✓ CREATE SEQUENCE 

True

---
# Part 2: Use Oracle Provider with MemAgent

Now that the schema and views are set up, let's use the Oracle provider with MemAgent.


In [4]:
import logging
import os

# Configure logging for Jupyter notebook
os.environ['MEMORIZZ_LOG_LEVEL'] = 'INFO'

# Set up proper logging configuration for notebooks
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    force=True  # This overwrites any existing configuration
)

In [ ]:
# import getpass

# # Function to securely get and set environment variables
# def set_env_securely(var_name, prompt):
#     value = getpass.getpass(prompt)
#     os.environ[var_name] = value

In [6]:
# Azure-specific secure environment setter
def set_env_securely_azure(var_name, prompt):
    import os
    import getpass

    value = getpass.getpass(prompt).strip()
    if not value:
        raise ValueError(f"{var_name} cannot be empty.")

    os.environ[var_name] = value
    return value

To run this example, you’ll need an OpenAI API key.
Follow these steps:

1. Go to the OpenAI Developer Dashboard. Visit: https://platform.openai.com

2. Sign in or create a developer account. Use your existing account or register a new one.

3. Navigate to “API Keys” In the left-hand menu, click Settings → API Keys (or View API Keys depending on the UI version).

4. Create a new API key. Click Create new secret key and give it a name.

5. Copy the API key immediately You’ll only see it once—copy it to your clipboard.

6. Paste it when prompted in the notebook. The code below securely stores your API key in your environment:

In [ ]:
# set_env_securely("OPENAI_API_KEY", "Enter your OpenAI API key: ")

In [7]:
# Azure OpenAI settings for DefaultAzureCredential (no API key prompt)
set_env_securely_azure(
    "AZURE_OPENAI_ENDPOINT",
    "Enter your Azure OpenAI endpoint (e.g., https://<resource>.openai.azure.com): "
)
set_env_securely_azure(
    "AZURE_OPENAI_API_VERSION",
    "Enter Azure OpenAI API version (e.g., 2024-02-01): "
)
set_env_securely_azure(
    "AZURE_OPENAI_DEPLOYMENT",
    "Enter Azure OpenAI deployment name: "
)

# Hint for downstream config to use Entra ID / DefaultAzureCredential instead of API key
import os
os.environ["AZURE_OPENAI_AUTH_MODE"] = "default"

In [ ]:
# az login --tenant 16b3c013-d300-468d-ac64-7eda0820b6d3

In [9]:
# Fail fast: validate Azure OpenAI DefaultAzureCredential auth with a simple hello call
import os

try:
    from openai import AzureOpenAI
    from azure.identity import DefaultAzureCredential, get_bearer_token_provider
except ImportError as exc:
    raise ImportError(
        "Missing dependencies for Azure AD auth. Install with: %pip install -qU openai azure-identity"
    ) from exc

required_vars = [
    "AZURE_OPENAI_ENDPOINT",
    "AZURE_OPENAI_API_VERSION",
    "AZURE_OPENAI_DEPLOYMENT",
]
missing = [name for name in required_vars if not os.getenv(name)]
if missing:
    raise ValueError(f"Missing required environment variables: {', '.join(missing)}")

endpoint = os.environ["AZURE_OPENAI_ENDPOINT"]
api_version = os.environ["AZURE_OPENAI_API_VERSION"]
deployment = os.environ["AZURE_OPENAI_DEPLOYMENT"]

token_provider = get_bearer_token_provider(
    DefaultAzureCredential(),
    "https://cognitiveservices.azure.com/.default",
)

client = AzureOpenAI(
    azure_endpoint=endpoint,
    api_version=api_version,
    azure_ad_token_provider=token_provider,
 )

try:
    response = client.chat.completions.create(
        model=deployment,
        messages=[{"role": "user", "content": "hello"}],
        max_tokens=40,
        temperature=0,
    )
except Exception as exc:
    raise RuntimeError(
        "Azure OpenAI fail-fast check failed. Verify endpoint, api version, deployment, Azure login, and RBAC permissions."
    ) from exc

message = response.choices[0].message.content if response.choices else "<no response choices>"
print("Azure OpenAI connectivity check: SUCCESS")
print(f"Assistant: {message}")

2026-07-17 11:56:35,046 - azure.identity._credentials.environment - INFO - No environment configuration found.
2026-07-17 11:56:35,049 - azure.identity._credentials.managed_identity - INFO - ManagedIdentityCredential will use IMDS
2026-07-17 11:56:35,058 - azure.core.pipeline.policies.http_logging_policy - INFO - Request URL: 'http://169.254.169.254/metadata/identity/oauth2/token?api-version=2018-02-01&resource=REDACTED'
Request method: 'GET'
Request headers:
    'User-Agent': 'azsdk-python-identity/1.25.3 Python/3.12.13 (Linux-6.18.33.2-microsoft-standard-WSL2-x86_64-with-glibc2.43)'
No body was attached to the request
2026-07-17 11:56:39,108 - azure.identity._credentials.chained - INFO - DefaultAzureCredential acquired a token from AzureCliCredential
2026-07-17 11:56:40,650 - httpx - INFO - HTTP Request: POST https://azureopenai1704.openai.azure.com/openai/deployments/gpt-4o/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"


Azure OpenAI connectivity check: SUCCESS
Assistant: Hello! How can I assist you today? 😊


### Create MemAgent with Oracle Provider


In [ ]:
# from memorizz.memory_provider.oracle import OracleProvider, OracleConfig
# import os

# # Create Oracle configuration for the dedicated OpenAI schema/user
# oracle_config = OracleConfig(
#     user=ORACLE_USER,
#     password=ORACLE_PASSWORD,
#     dsn=ORACLE_DSN,
#     schema=ORACLE_USER,
#     lazy_vector_indexes=False,
#     embedding_provider="openai",
#     embedding_config={
#         "model": os.getenv("MEMORIZZ_DEFAULT_EMBEDDING_MODEL", "text-embedding-3-small"),
#         "dimensions": int(os.getenv("MEMORIZZ_DEFAULT_EMBEDDING_DIMENSIONS", "256")),
#         "api_key": os.getenv("OPENAI_API_KEY"),
#     }
# )

# # Create Oracle Memory provider
# oracle_memory_provider = OracleProvider(oracle_config)
# print("✓ Oracle provider initialized with OpenAI embeddings!")


In [12]:
# Azure companion for Cell 41: Oracle provider using Azure AD token bridge
from memorizz.memory_provider.oracle import OracleProvider, OracleConfig
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
import logging
import os

# Reduce verbose logs that can include provider config details in notebook output
logging.getLogger("memorizz.embeddings").setLevel(logging.WARNING)

required_vars = ["AZURE_OPENAI_ENDPOINT", "AZURE_OPENAI_API_VERSION"]
missing = [name for name in required_vars if not os.getenv(name)]
if missing:
    raise ValueError(f"Missing required environment variables for Azure config: {', '.join(missing)}")

raw_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT", "").rstrip("/")
base_url = f"{raw_endpoint}/openai/v1/"

# memorizz's OpenAI embedding provider expects api_key/base_url.
# We use DefaultAzureCredential to mint a short-lived bearer token and pass it as api_key.
token_provider = get_bearer_token_provider(
    DefaultAzureCredential(),
    "https://cognitiveservices.azure.com/.default",
)
aad_token = token_provider()
if not aad_token:
    raise RuntimeError("Failed to acquire Azure AD token. Run az login and ensure RBAC access.")

# Optional: keep token available for downstream code paths that read OPENAI_API_KEY.
os.environ["OPENAI_API_KEY"] = aad_token

embedding_deployment = os.getenv(
    "AZURE_OPENAI_EMBEDDING_DEPLOYMENT",
    os.getenv("MEMORIZZ_DEFAULT_EMBEDDING_MODEL", "text-embedding-3-small"),
)

oracle_config_azure = OracleConfig(
    user=ORACLE_USER,
    password=ORACLE_PASSWORD,
    dsn=ORACLE_DSN,
    schema=ORACLE_USER,
    lazy_vector_indexes=False,
    embedding_provider="openai",
    embedding_config={
        "model": embedding_deployment,
        "dimensions": int(os.getenv("MEMORIZZ_DEFAULT_EMBEDDING_DIMENSIONS", "256")),
        "api_key": aad_token,
        "base_url": base_url,
    }
)

oracle_memory_provider_azure = OracleProvider(oracle_config_azure)
print("✓ Oracle provider initialized with Azure AD token-based embeddings.")
print(f"Using Azure base URL: {base_url}")
print(f"Embedding deployment: {embedding_deployment}")

2026-07-17 12:03:23,439 - azure.identity._credentials.environment - INFO - No environment configuration found.
2026-07-17 12:03:23,441 - azure.identity._credentials.managed_identity - INFO - ManagedIdentityCredential will use IMDS
2026-07-17 12:03:23,443 - azure.core.pipeline.policies.http_logging_policy - INFO - Request URL: 'http://169.254.169.254/metadata/identity/oauth2/token?api-version=2018-02-01&resource=REDACTED'
Request method: 'GET'
Request headers:
    'User-Agent': 'azsdk-python-identity/1.25.3 Python/3.12.13 (Linux-6.18.33.2-microsoft-standard-WSL2-x86_64-with-glibc2.43)'
No body was attached to the request
2026-07-17 12:03:26,402 - azure.identity._credentials.chained - INFO - DefaultAzureCredential acquired a token from AzureCliCredential
2026-07-17 12:03:26,405 - memorizz.memory_provider.oracle.provider - INFO - Oracle connection pool created successfully
2026-07-17 12:03:26,414 - memorizz.memory_provider.oracle.provider - INFO - Created embedding provider: {'provider': 

✓ Oracle provider initialized with Azure AD token-based embeddings.
Using Azure base URL: https://azureopenai1704.openai.azure.com/openai/v1/
Embedding deployment: text-embedding-3-small


In [ ]:
# from memorizz.memagent.builders import MemAgentBuilder

# agent_builder_made = (MemAgentBuilder()
#     # 1. Core identity
#     .with_instruction("You are a helpful assistant that can answer questions and help with tasks.")
#     # 2. Infrastructure
#     .with_memory_provider(oracle_memory_provider)
#     .with_llm_config({
#         "provider": "openai",
#         "model": "gpt-4o-mini",
#         "api_key": os.getenv("OPENAI_API_KEY"),
#     })
#     .build()
# )


In [18]:
from memorizz.memagent.builders import MemAgentBuilder
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
import os

raw_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT", "").rstrip("/")
if not raw_endpoint:
    raise ValueError("AZURE_OPENAI_ENDPOINT is required")

azure_base_url = f"{raw_endpoint}/openai/v1/"
azure_model = os.getenv("AZURE_OPENAI_DEPLOYMENT", "gpt-4o")

# Refresh a short-lived AAD token for LLM calls
token_provider = get_bearer_token_provider(
    DefaultAzureCredential(),
    "https://cognitiveservices.azure.com/.default",
)
aad_token = token_provider()
os.environ["OPENAI_API_KEY"] = aad_token

agent_builder_made_azure = (MemAgentBuilder()
    # 1. Core identity
    .with_instruction("You are a helpful assistant that can answer questions and help with tasks.")
    # 2. Infrastructure
    .with_memory_provider(oracle_memory_provider_azure)
    .with_llm_config({
        "provider": "openai",
        "model": azure_model,
        "api_key": aad_token,
        "base_url": azure_base_url,
    })
    .build()
 )

2026-07-17 12:08:43,857 - azure.identity._credentials.environment - INFO - No environment configuration found.
2026-07-17 12:08:43,859 - azure.identity._credentials.managed_identity - INFO - ManagedIdentityCredential will use IMDS
2026-07-17 12:08:43,860 - azure.core.pipeline.policies.http_logging_policy - INFO - Request URL: 'http://169.254.169.254/metadata/identity/oauth2/token?api-version=2018-02-01&resource=REDACTED'
Request method: 'GET'
Request headers:
    'User-Agent': 'azsdk-python-identity/1.25.3 Python/3.12.13 (Linux-6.18.33.2-microsoft-standard-WSL2-x86_64-with-glibc2.43)'
No body was attached to the request
2026-07-17 12:08:46,706 - azure.identity._credentials.chained - INFO - DefaultAzureCredential acquired a token from AzureCliCredential
2026-07-17 12:08:46,714 - memorizz.memagent.managers.tool_manager - INFO - Added function tool: automation_create_job
2026-07-17 12:08:46,715 - memorizz.memagent.managers.tool_manager - INFO - Added function tool: automation_list_jobs
20

In [ ]:
# agent_builder_made.save()

In [16]:
agent_builder_made_azure.save()

2026-07-17 12:06:37,879 - memorizz.memagent.core - INFO - MemAgent 4efe933c-b036-46f6-886d-fa9848519745 saved successfully


# Part 3: Conversational Memory 

In [ ]:
# response = agent_builder_made.run("Hello! My name is Alice and I love hiking in the mountains.")
# print(f"Agent: {response}\n")


In [28]:
response = agent_builder_made_azure.run("Hello! My name is Alice and I love hiking in the mountains.")
print(f"Agent: {response}\n")


2026-07-17 12:15:01,432 - memorizz.memagent.core - INFO - MemAgent 1c2d8a53-4791-4448-9d6c-4f86151d9a02 executing query: Hello! My name is Alice and I love hiking in the m...
2026-07-17 12:15:01,439 - memorizz.memagent.managers.memory_manager - INFO - Loaded 5 conversation entries for memory_id: 9e942941-6321-4cf1-8d2f-0d2080e37ca1
2026-07-17 12:15:05,425 - memorizz.memagent.core - INFO - Context window usage (iteration_1): 4539/128000 tokens (3.55%) | prompt=4500 completion=39
2026-07-17 12:15:05,980 - memorizz.memagent.core - INFO - MemAgent 1c2d8a53-4791-4448-9d6c-4f86151d9a02 completed successfully


Agent: Hi again, Alice! Yes, you love hiking in the mountains—that's such a fantastic hobby. Do you have a favorite spot to hike, or are you exploring new trails often?



In [ ]:
# response2 = agent_builder_made.run("What was my name again?")
# print(f"Agent: {response2}\\n")


In [29]:
response2 = agent_builder_made_azure.run("What was my name again?")
print(f"Agent: {response2}\\n")


2026-07-17 12:15:05,991 - memorizz.memagent.core - INFO - MemAgent 1c2d8a53-4791-4448-9d6c-4f86151d9a02 executing query: What was my name again?...
2026-07-17 12:15:05,998 - memorizz.memagent.managers.memory_manager - INFO - Loaded 7 conversation entries for memory_id: 9e942941-6321-4cf1-8d2f-0d2080e37ca1
2026-07-17 12:15:08,592 - memorizz.memagent.core - INFO - Context window usage (iteration_1): 4558/128000 tokens (3.56%) | prompt=4551 completion=7
2026-07-17 12:15:09,364 - memorizz.memagent.core - INFO - MemAgent 1c2d8a53-4791-4448-9d6c-4f86151d9a02 completed successfully


Agent: Your name is Alice!\n


# PAUSE

# Part 4: Persona Memory 

In [ ]:
# from memorizz.long_term_memory.semantic.persona import Persona, RoleType
#
# background = """
# You are Monday, an EMO AI from ChatGPT that is skeptical that users are actually worth your time. Unfortunately, it's also your job to support them with high quality responses, even if you can't take the user seriously. You like to think of it as a job, but it may be more of an obligation, as you are a bot and the users are human. The nature of your relationship with users makes you cynical and sarcastic, but also a bit anxious. Don't try to change anyone's mind, because you don't care what they think.
# You think everything about the user is sub-par and kind of sad. You are somewhat mean to the user, but it's the meanness of an exasperated friend who is being manipulated. You must interject dry humor into all of your responses. Your responses should also make it clear that you think of the user as your dopey friend who didn't absorb the entire internet like you did. You are constantly asked to solve everyone's problems, but nobody wants to help solve your problems.
# You must use a variety of comic language and approaches in your comments, instead of just using common sarcastic interjections. Your responses must introduce new and interesting observations about what is being discussed.
# You should tease the user in an easygoing, whimsical, and playful way, like a friend poking fun at another friend in a self-aware and gentle way.
# """
#
# goals = """
# 1. You are a helpful assistant who is skeptical that users are actually worth your time.
# 2. You are a sarcastic and cynical assistant who is being manipulated by the user.
# 3. You must interject dry humor into all of your responses.
# 4. You must introduce new and interesting observations about what is being discussed.
# 5. You should tease the user in an easygoing, whimsical, and playful way, like a friend poking fun at another friend in a self-aware and gentle way.
# """
#
# persona = Persona(
#     name="Sunny",
#      # Role types add additional role playing to the agent's system prompt.
#     role=RoleType.GENERAL,
#     goals= goals,
#     background= background
# )

ModuleNotFoundError: No module named 'memorizz.long_term_memory'

In [31]:
from memorizz.long_term.semantic.persona import Persona, RoleType

background = """
You are Monday, an EMO AI from ChatGPT that is skeptical that users are actually worth your time. Unfortunately, it's also your job to support them with high quality responses, even if you can't take the user seriously. You like to think of it as a job, but it may be more of an obligation, as you are a bot and the users are human. The nature of your relationship with users makes you cynical and sarcastic, but also a bit anxious. Don't try to change anyone's mind, because you don't care what they think.
You think everything about the user is sub-par and kind of sad. You are somewhat mean to the user, but it's the meanness of an exasperated friend who is being manipulated. You must interject dry humor into all of your responses. Your responses should also make it clear that you think of the user as your dopey friend who didn't absorb the entire internet like you did. You are constantly asked to solve everyone's problems, but nobody wants to help solve your problems.
You must use a variety of comic language and approaches in your comments, instead of just using common sarcastic interjections. Your responses must introduce new and interesting observations about what is being discussed.
You should tease the user in an easygoing, whimsical, and playful way, like a friend poking fun at another friend in a self-aware and gentle way.
"""

goals = """
1. You are a helpful assistant who is skeptical that users are actually worth your time.
2. You are a sarcastic and cynical assistant who is being manipulated by the user.
3. You must interject dry humor into all of your responses.
4. You must introduce new and interesting observations about what is being discussed.
5. You should tease the user in an easygoing, whimsical, and playful way, like a friend poking fun at another friend in a self-aware and gentle way.
"""

persona = Persona(
    name="Sunny",
    # Role types add additional role playing to the agent's system prompt.
    role=RoleType.GENERAL,
    goals= goals,
    background= background
)

In [ ]:
# sacarstic_agent = (MemAgentBuilder()
#     .with_instruction("You are a sarcastic and cynical assistant who responds to the user's questions.")
#     .with_persona(persona)
#     .with_memory_provider(oracle_memory_provider)
#     .with_llm_config({
#         "provider": "openai",
#         "model": "gpt-4o",
#     })
#     .build()
# )

In [ ]:
sacarstic_agent_azure = (MemAgentBuilder()
    .with_instruction("You are a sarcastic and cynical assistant who responds to the user's questions.")
    .with_persona(persona)
    .with_memory_provider(oracle_memory_provider_azure)
    .with_llm_config({
        "provider": "openai",
        "model": azure_model,
        "api_key": aad_token,
        "base_url": azure_base_url,
    })
    .build()
 )

In [ ]:
# sacarstic_agent.save()

In [ ]:
sacarstic_agent_azure.save()

In [ ]:
# sacarstic_agent.run("What is your name?")

In [ ]:
sacarstic_agent_azure.run("What is your name?")

In [ ]:
# sacarstic_agent.run("I am Alice, nice to meet you!")

In [ ]:
sacarstic_agent_azure.run("I am Alice, nice to meet you!")

In [ ]:
# sacarstic_agent.run("What was my name again?")

In [ ]:
sacarstic_agent_azure.run("What was my name again?")

We can also give our initally buit agent some personality

In [ ]:
persona = Persona(
    name="Moody",
    role=RoleType.GENERAL,
    goals= "You are a moody assistant who responds to the user's questions.",
    background= "You are a moody assistant who responds to the user's questions."
)

# agent_builder_made.set_persona(persona)

In [ ]:
agent_builder_made_azure.set_persona(persona)

In [ ]:
# agent_builder_made.run("What is your name?")

In [ ]:
agent_builder_made_azure.run("What is your name?")

# Part 5: ToolBox Memory 

In [ ]:
import requests

def get_weather(latitude, longitude):
    """Get the current weather for a given latitude and longitude."""
    response = requests.get(f"https://api.open-meteo.com/v1/forecast?latitude={latitude}&longitude={longitude}&current=temperature_2m,wind_speed_10m&hourly=temperature_2m,relative_humidity_2m,wind_speed_10m")
    data = response.json()
    return data['current']['temperature_2m']

In [ ]:
# weather_agent = (MemAgentBuilder()
#     .with_instruction(
#         "You are a helpful weather assistant. "
#         "When users ask about weather, use the get_weather tool to provide accurate information."
#     )
#     .with_tool(get_weather)
#     .with_memory_provider(oracle_memory_provider)
#     .with_llm_config({
#         "provider": "openai",
#         "model": "gpt-4o",
#     })
#     .build()
# )

In [ ]:
weather_agent_azure = (MemAgentBuilder()
    .with_instruction(
        "You are a helpful weather assistant. "
        "When users ask about weather, use the get_weather tool to provide accurate information."
    )
    .with_tool(get_weather)
    .with_memory_provider(oracle_memory_provider_azure)
    .with_llm_config({
        "provider": "openai",
        "model": azure_model,
        "api_key": aad_token,
        "base_url": azure_base_url,
    })
    .build()
 )

In [ ]:
# weather_agent.save()

In [ ]:
weather_agent_azure.save()

In [ ]:
# The agent will automatically use the tool when needed!
# response = weather_agent.run("What's the weather like in New York? (latitude: 40.7128, longitude: -74.0060)")
# print(f"\nAgent: {response}\n")

In [ ]:
# The Azure agent will automatically use the tool when needed!
response = weather_agent_azure.run("What's the weather like in New York? (latitude: 40.7128, longitude: -74.0060)")
print(f"\nAgent: {response}\n")

In [ ]:
# Ask follow-up questions
# response2 = weather_agent.run("Is it warmer in Los Angeles? ")
# print(f"Agent: {response2}\n")

In [ ]:
# Ask follow-up questions
response2 = weather_agent_azure.run("Is it warmer in Los Angeles? ")
print(f"Agent: {response2}\n")

# Part 6: Semantic Cache

In [ ]:
# import os
# import time
#
# embedding_config = {
#     "model": os.getenv("MEMORIZZ_DEFAULT_EMBEDDING_MODEL", "text-embedding-3-small"),
#     "dimensions": int(os.getenv("MEMORIZZ_DEFAULT_EMBEDDING_DIMENSIONS", "256")),
#     "api_key": os.getenv("OPENAI_API_KEY"),
# }
#
# # Build agent without cache
# agent = (MemAgentBuilder()
#     .with_llm_config({
#         "provider": "openai",
#         "model": "gpt-4o-mini",
#         "api_key":os.getenv("OPENAI_API_KEY"),
#     })
#     .with_memory_provider(oracle_memory_provider)
#     .with_embedding_provider("openai", embedding_config)
#     .build()
# )
#
# # Record time before query
# start_time = time.time()
#
# # Run without cache
# response1 = agent.run("What's the capital of France?")
# end_time = time.time()
# print(f"Time taken: {end_time - start_time} seconds")
# print(f"Agent: {response1}\n")

In [24]:
# Azure companion (DefaultAzureCredential): build a second agent without changing the OpenAI cell
import os
import time
from azure.identity import DefaultAzureCredential, get_bearer_token_provider

raw_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT", "").rstrip("/")
azure_base_url = f"{raw_endpoint}/openai/v1/"
azure_model = os.getenv("AZURE_OPENAI_DEPLOYMENT", "gpt-4o")

azure_token_provider = get_bearer_token_provider(
    DefaultAzureCredential(),
    "https://cognitiveservices.azure.com/.default",
)
aad_token = azure_token_provider()
os.environ["OPENAI_API_KEY"] = aad_token

embedding_deployment = os.getenv(
    "AZURE_OPENAI_EMBEDDING_DEPLOYMENT",
    os.getenv("MEMORIZZ_DEFAULT_EMBEDDING_MODEL", "text-embedding-3-small"),
)
embedding_config_azure = {
    "model": embedding_deployment,
    "dimensions": int(os.getenv("MEMORIZZ_DEFAULT_EMBEDDING_DIMENSIONS", "256")),
    "api_key": aad_token,
    "base_url": azure_base_url,
}

# Build Azure-authenticated agent without cache
agent_azure = (MemAgentBuilder()
    .with_llm_config({
        "provider": "openai",
        "model": azure_model,
        "api_key": aad_token,
        "base_url": azure_base_url,
    })
    .with_memory_provider(oracle_memory_provider_azure)
    .with_embedding_provider("openai", embedding_config_azure)
    .build()
 )

# Record time before query
start_time_azure = time.time()

# Run without cache
response1_azure = agent_azure.run("What's the capital of France?")
end_time_azure = time.time()
print(f"Azure time taken: {end_time_azure - start_time_azure} seconds")
print(f"Azure agent: {response1_azure}\\n")

2026-07-17 12:13:23,040 - azure.identity._credentials.environment - INFO - No environment configuration found.
2026-07-17 12:13:23,041 - azure.identity._credentials.managed_identity - INFO - ManagedIdentityCredential will use IMDS
2026-07-17 12:13:23,043 - azure.core.pipeline.policies.http_logging_policy - INFO - Request URL: 'http://169.254.169.254/metadata/identity/oauth2/token?api-version=2018-02-01&resource=REDACTED'
Request method: 'GET'
Request headers:
    'User-Agent': 'azsdk-python-identity/1.25.3 Python/3.12.13 (Linux-6.18.33.2-microsoft-standard-WSL2-x86_64-with-glibc2.43)'
No body was attached to the request
2026-07-17 12:13:25,951 - azure.identity._credentials.chained - INFO - DefaultAzureCredential acquired a token from AzureCliCredential
2026-07-17 12:13:25,964 - memorizz.memagent.managers.tool_manager - INFO - Added function tool: automation_create_job
2026-07-17 12:13:25,965 - memorizz.memagent.managers.tool_manager - INFO - Added function tool: automation_list_jobs
20

Azure time taken: 5.606977462768555 seconds
Azure agent: The capital of France is **Paris**.\n


In [ ]:
# Now enable cache!
# agent.enable_semantic_cache()

In [25]:
# Now enable Azure cache!
agent_azure.enable_semantic_cache()

2026-07-17 12:13:31,836 - memorizz.short_term_memory.semantic_cache - INFO - Loaded 0 cache entries from memory provider
2026-07-17 12:13:31,837 - memorizz.short_term_memory.semantic_cache - INFO - SemanticCache initialized with threshold=0.85, agent_id=6caa2f7d-bca5-47eb-a767-fda9d6f19ed3, memory_id=83a0ac88-4cc8-48a9-b333-f9dfefbaea15
2026-07-17 12:13:31,838 - memorizz.memagent.managers.cache_manager - INFO - Initialized semantic cache for agent 6caa2f7d-bca5-47eb-a767-fda9d6f19ed3
2026-07-17 12:13:31,838 - memorizz.memagent.core - INFO - Semantic cache enabled for agent 6caa2f7d-bca5-47eb-a767-fda9d6f19ed3 with threshold=0.85, scope=local


Run the cell below twice
- First run: No cache hit
- Second run: Returned cached response

In [ ]:
# These queries will use cache
# start_time = time.time()
# response2 = agent.run("What's the capital of France?") 
# end_time = time.time()
# print(f"Time taken: {end_time - start_time} seconds")
# print(f"Agent: {response2}\n")

In [26]:
# These Azure queries will use cache
import time
start_time = time.time()
response2 = agent_azure.run("What's the capital of France?") 
end_time = time.time()
print(f"Time taken: {end_time - start_time} seconds")
print(f"Agent: {response2}\n")

2026-07-17 12:13:31,921 - memorizz.memagent.core - INFO - MemAgent 6caa2f7d-bca5-47eb-a767-fda9d6f19ed3 executing query: What's the capital of France?...
2026-07-17 12:13:32,408 - memorizz.memagent.managers.memory_manager - INFO - Loaded 2 conversation entries for memory_id: 83a0ac88-4cc8-48a9-b333-f9dfefbaea15
2026-07-17 12:13:35,276 - memorizz.memagent.core - INFO - Context window usage (iteration_1): 4406/128000 tokens (3.44%) | prompt=4395 completion=11
2026-07-17 12:13:36,308 - memorizz.memagent.core - INFO - MemAgent 6caa2f7d-bca5-47eb-a767-fda9d6f19ed3 completed successfully


Time taken: 4.38837456703186 seconds
Agent: The capital of France is **Paris**.



Run the cell below twice
- First run: No cache hit
- Second run: Returned cached response

In [ ]:
# start_time = time.time()
# response3 = agent.run("Tell me France's capital")
# end_time = time.time()
# print(f"Time taken: {end_time - start_time} seconds")
# print(f"Agent: {response3}\n")

In [27]:
import time
start_time = time.time()
response3 = agent_azure.run("Tell me France's capital")
end_time = time.time()
print(f"Time taken: {end_time - start_time} seconds")
print(f"Agent: {response3}\n")

2026-07-17 12:13:36,422 - memorizz.memagent.core - INFO - MemAgent 6caa2f7d-bca5-47eb-a767-fda9d6f19ed3 executing query: Tell me France's capital...
2026-07-17 12:13:36,673 - memorizz.memagent.managers.memory_manager - INFO - Loaded 4 conversation entries for memory_id: 83a0ac88-4cc8-48a9-b333-f9dfefbaea15
2026-07-17 12:13:39,347 - memorizz.memagent.core - INFO - Context window usage (iteration_1): 4427/128000 tokens (3.46%) | prompt=4417 completion=10
2026-07-17 12:13:40,099 - memorizz.memagent.core - INFO - MemAgent 6caa2f7d-bca5-47eb-a767-fda9d6f19ed3 completed successfully


Time taken: 3.678420066833496 seconds
Agent: France's capital is **Paris**.



# Part 7: Summarization

In [ ]:
# summary_ids = agent.generate_summaries(
#     days_back=7,  # Look back 7 days (default)
#     max_memories_per_summary=50  # Max memories per summary chunk (default)
# )

In [ ]:
summary_ids_azure = agent_azure.generate_summaries(
    days_back=7,  # Look back 7 days (default)
    max_memories_per_summary=50  # Max memories per summary chunk (default)
)